In [ ]:
import pyodbc
import pandas as pd
import duckdb
import json
import os

# =========================================================
# CONFIGURACIÓN
# =========================================================
CONFIG_PATH = os.path.join("..", "config.json")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

server   = config["SERVER"]
database = config["DATABASE"]
username = config["DB_USER"]
password = config["DB_PASS"]

DUCKDB_ITEMS   = "items_referencias.duckdb"
DUCKDB_PRECIOS = "precios.duckdb"

EXCEL_OUT = "items_referencias_con_precios.xlsx"

TABLE_ITEMS = "items_referencias"
TABLE_FINAL = "items_referencias_con_precios"

print("✅ Configuración cargada")

# =========================================================
# SQL SERVER – QUERY ORIGINAL (ÍNTEGRO)
# =========================================================
SQL_QUERY = """
SELECT DISTINCT
    i.f120_id                   AS Item,
    i.f120_referencia           AS Referencia_Principal,
    r.f124_referencia           AS Referencia_Alterna,
    i.f120_descripcion          AS Descripcion_Item,
    i.f120_id_unidad_inventario AS UM,
    ie.f121_notas               AS Notas,
    CASE ie.f121_ind_estado
        WHEN 1 THEN 'Activo'
        WHEN 0 THEN 'Inactivo'
        ELSE 'Bloqueado'
    END AS Estado
FROM t120_mc_items i
INNER JOIN t121_mc_items_extensiones ie
    ON ie.f121_rowid_item = i.f120_rowid
LEFT JOIN t124_mc_items_referencias r
    ON r.f124_rowid_item = i.f120_rowid
WHERE i.f120_id_cia = 1
AND EXISTS (
    SELECT 1
    FROM t125_mc_items_criterios c
    WHERE c.f125_rowid_item = i.f120_rowid
      AND c.f125_id_plan = '07'
      AND c.f125_id_criterio_mayor = '051'
)
AND NOT EXISTS (
    SELECT 1
    FROM t058_mm_usuario_entidad ue
    WHERE ue.f058_id_cia = 1
      AND ue.f058_entidad = 149
      AND ue.f058_ind_aplica_consultas = 1
      AND ue.f058_id_tipo_inv_serv = i.f120_id_tipo_inv_serv
)
ORDER BY i.f120_id;
"""

# =========================================================
# CONEXIÓN SQL SERVER
# =========================================================
connection = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
)

print("✅ Conectado a SQL Server")

# =========================================================
# CONEXIÓN DUCKDB
# =========================================================
duck = duckdb.connect(DUCKDB_ITEMS)
duck.execute(f"DROP TABLE IF EXISTS {TABLE_ITEMS}")

# =========================================================
# CREAR TABLA BASE VACÍA
# =========================================================
duck.execute(f"""
CREATE TABLE {TABLE_ITEMS} AS
SELECT
    CAST(NULL AS INTEGER) AS Item,
    CAST(NULL AS VARCHAR) AS Referencia_Principal,
    CAST(NULL AS VARCHAR) AS Referencia_Alterna,
    CAST(NULL AS VARCHAR) AS Descripcion_Item,
    CAST(NULL AS VARCHAR) AS UM,
    CAST(NULL AS VARCHAR) AS Notas,
    CAST(NULL AS VARCHAR) AS Estado
WHERE 1=0;
""")

print("✅ Tabla base creada")

# =========================================================
# EXTRACCIÓN POR CHUNKS (SIN DUPLICAR)
# =========================================================
print("▶ Cargando datos en DuckDB...")

for chunk in pd.read_sql(SQL_QUERY, connection, chunksize=50_000):
    duck.register("tmp_df", chunk)

    duck.execute(f"""
        INSERT INTO {TABLE_ITEMS}
        SELECT * FROM tmp_df
    """)

# =========================================================
# BLINDAJE DE UNICIDAD
# =========================================================
duck.execute(f"""
CREATE UNIQUE INDEX ux_ref_principal_alterna
ON {TABLE_ITEMS} (Referencia_Principal, Referencia_Alterna);
""")

print("✅ Unicidad garantizada")

# =========================================================
# LIMPIEZA Y NORMALIZACIÓN (ALTERNAS)
# =========================================================
duck.execute(f"""
ALTER TABLE {TABLE_ITEMS}
ADD COLUMN Original VARCHAR;
""")

duck.execute(f"""
UPDATE {TABLE_ITEMS}
SET Original =
    trim(
        replace(
            replace(Referencia_Alterna, chr(10), ''),
            chr(13), ''
        )
    );
""")

duck.execute(f"""
ALTER TABLE {TABLE_ITEMS}
ADD COLUMN Referencia VARCHAR;
""")

duck.execute(f"""
UPDATE {TABLE_ITEMS}
SET Referencia =
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(
                            replace(Original, '_', '-'),
                            '[^A-Za-z0-9\\.\\-"/ ]',
                            '',
                            'g'
                        ),
                        '^\\.+|\\.+$',
                        '',
                        'g'
                    ),
                    '\\.{2,}',
                    '.',
                    'g'
                ),
                '\\s*-\\s*',
                '-',
                'g'
            ),
            '[ ]{3,}',
            ' ',
            'g'
        )
    );
""")

print("✅ Referencia alterna normalizada")


'''
# =========================================================
# LIMPIEZA Y NORMALIZACIÓN (ALTERNAS)
# =========================================================
duck.execute(f"""
ALTER TABLE {TABLE_ITEMS}
ADD COLUMN Original VARCHAR;
""")

duck.execute(f"""
UPDATE {TABLE_ITEMS}
SET Original =
    trim(
        replace(
            replace(Referencia_Alterna, chr(10), ''),
            chr(13), ''
        )
    );
""")

duck.execute(f"""
ALTER TABLE {TABLE_ITEMS}
ADD COLUMN Referencia VARCHAR;
""")

duck.execute(f"""
UPDATE {TABLE_ITEMS}
SET Referencia =
    regexp_replace(
        regexp_replace(
            replace(Original, '_', '-'),
            '[^A-Za-z0-9\\-"/]',
            '',
            'g'
        ),
        '[-]{2,}',
        '-',
        'g'
    );
""")

print("✅ Referencia normalizada desde alterna")

'''

# =========================================================
# CRUCE CON PRECIOS (POR ALTERNA NORMALIZADA)
# =========================================================
duck.execute(f"ATTACH '{DUCKDB_PRECIOS}' AS precios;")

duck.execute(f"""
CREATE OR REPLACE TABLE {TABLE_FINAL} AS
SELECT
    i.*,
    p.precio_usa,
    p.precio_br,
    p.precio_eur,
    p.disp_br,
    p.disp_eur,
    p.disp_usa

FROM {TABLE_ITEMS} i
LEFT JOIN precios.precios_consolidados p
    ON i.Referencia = p.referencia;
""")

print("✅ Precios agregados correctamente")

# =========================================================
# EXPORTAR A EXCEL
# =========================================================
df_out = duck.execute(f"SELECT * FROM {TABLE_FINAL}").df()
df_out.to_excel(EXCEL_OUT, index=False)

print(f"📂 Excel generado: {EXCEL_OUT}")

# =========================================================
# CIERRES
# =========================================================
duck.close()
connection.close()

print("✅ Proceso finalizado correctamente")

In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
from pathlib import Path
import pandas as pd
import duckdb

# =========================================================
# 0.1 FACTORES DE IMPORTACIÓN
# =========================================================
FACTOR_BR  = 1.10
FACTOR_USA = 1.25
FACTOR_EUR = 1.30

# =========================================================
# 1. RUTAS
# =========================================================
DUCKDB_MAIN    = "precio_de_lista.duckdb"
DUCKDB_ITEMS   = "items_referencias.duckdb"
DUCKDB_PRECIOS = "precios.duckdb"

EXCEL_LISTA_PATH = Path(
    r"C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Precios De Lista_Prueba.xlsx"
)

EXCEL_OUT = "Precio_De_Lista_con_Precios.xlsx"

# =========================================================
# 2. CONEXIÓN + ATTACH
# =========================================================
duck = duckdb.connect(DUCKDB_MAIN)
duck.execute(f"ATTACH '{DUCKDB_ITEMS}'   AS items_db;")
duck.execute(f"ATTACH '{DUCKDB_PRECIOS}' AS precios_db;")

print("✅ DuckDB conectado y bases adjuntadas")

# =========================================================
# 3. TABLA PRECIO_FAMILIA (REFERENCIA GANADORA)
# =========================================================
duck.execute(f"""
CREATE OR REPLACE TABLE precio_familia AS
WITH base AS (
    SELECT
        Item,
        Referencia_Principal,
        Original,
        Referencia,
        precio_usa,
        precio_br,
        precio_eur,
        disp_usa,
        disp_br,
        disp_eur,

        COALESCE(disp_usa,0)+COALESCE(disp_br,0)+COALESCE(disp_eur,0) AS suma_disponibilidad,
        COALESCE(precio_usa,0)+COALESCE(precio_br,0)+COALESCE(precio_eur,0) AS suma_precios,

        CASE
            WHEN precio_br IS NULL AND precio_usa IS NULL AND precio_eur IS NULL THEN NULL
            ELSE list_median(
                list_filter(
                    [
                        precio_br  * {FACTOR_BR},
                        precio_usa * {FACTOR_USA},
                        precio_eur * {FACTOR_EUR}
                    ],
                    x -> x IS NOT NULL
                )
            )
        END AS precio_rep
    FROM items_db.items_referencias_con_precios
),

familia AS (
    SELECT
        UPPER(TRIM(Referencia_Principal)) AS Referencia_Principal,
        COUNT(*) AS num_ref_activas,
        MAX(precio_rep) / NULLIF(MIN(NULLIF(precio_rep,0)),0) AS ratio_precio,

        '(' || string_agg(UPPER(TRIM(Original)), ',') || ')' AS RefsAlternas,

        '(' || string_agg(
            '(' ||
              COALESCE(CAST(ROUND(precio_br,2)  AS VARCHAR),'') || ',' ||
              COALESCE(CAST(ROUND(precio_usa,2) AS VARCHAR),'') || ',' ||
              COALESCE(CAST(ROUND(precio_eur,2) AS VARCHAR),'') ||
            ')', ','
        ) || ')' AS Precios_BR_USA_EURO,

        '(' || string_agg(
            '(' ||
              COALESCE(CAST(CAST(disp_br  AS BIGINT) AS VARCHAR),'') || ',' ||
              COALESCE(CAST(CAST(disp_usa AS BIGINT) AS VARCHAR),'') || ',' ||
              COALESCE(CAST(CAST(disp_eur AS BIGINT) AS VARCHAR),'') ||
            ')', ','
        ) || ')' AS Dispon_BR_USA_EURO
    FROM base
    GROUP BY UPPER(TRIM(Referencia_Principal))
),

ranked AS (
    SELECT
        b.*,
        f.num_ref_activas,
        f.ratio_precio,
        f.RefsAlternas,
        f.Precios_BR_USA_EURO,
        f.Dispon_BR_USA_EURO,

        ROW_NUMBER() OVER (
            PARTITION BY UPPER(TRIM(b.Referencia_Principal))
            ORDER BY
                b.suma_disponibilidad DESC,
                b.suma_precios DESC,
                CASE
                    WHEN UPPER(TRIM(b.Original)) = UPPER(TRIM(b.Referencia_Principal)) THEN 1
                    ELSE 0
                END DESC
        ) AS rn
    FROM base b
    JOIN familia f
      ON UPPER(TRIM(b.Referencia_Principal)) = f.Referencia_Principal
)

SELECT
    UPPER(TRIM(Referencia_Principal)) AS Referencia_Principal,
    Item,
    UPPER(TRIM(Original)) AS Referencia_Ganadora,
    UPPER(TRIM(Referencia)) AS Referencia_Normalizada_Ganadora,
    precio_usa,
    precio_br,
    precio_eur,
    disp_usa,
    disp_br,
    disp_eur,
    suma_precios,
    suma_disponibilidad,
    num_ref_activas,
    ratio_precio,
    RefsAlternas,
    Precios_BR_USA_EURO,
    Dispon_BR_USA_EURO
FROM ranked
WHERE rn = 1;
""")

print("✅ precio_familia creada")

# =========================================================
# 4. CARGA EXCEL PRECIO DE LISTA
# =========================================================
df_lista = pd.read_excel(EXCEL_LISTA_PATH)

duck.execute("DROP TABLE IF EXISTS precios_lista")
duck.register("tmp_precios_lista", df_lista)

duck.execute("""
CREATE TABLE precios_lista AS
SELECT
    UPPER(TRIM(Referencia)) AS ref_lista,
    *
FROM tmp_precios_lista;
""")

print("✅ Precios de Lista cargados")

# =========================================================
# 5. CRUCE FINAL + PRECIO PRORRATEADO
# =========================================================
duck.execute("""
CREATE OR REPLACE TABLE resultado_precios_lista AS
SELECT
    pl.*,

    pf.Referencia_Principal,
    pf.Referencia_Ganadora as Referencia_Activa,

    pf.precio_br  AS "Precio Brasil",
    pf.precio_usa AS "Precio Usa",
    pf.precio_eur AS "Precio Europa",

    pf.disp_br,
    pf.disp_usa,
    pf.disp_eur,

    pf.suma_precios,
    pf.suma_disponibilidad,

    pf.num_ref_activas,
    pf.ratio_precio,
    pf.RefsAlternas,
    pf.Precios_BR_USA_EURO,
    pf.Dispon_BR_USA_EURO,

    CASE
        WHEN pf.Referencia_Principal IS NOT NULL THEN 'PRINCIPAL'
        ELSE 'NO_MATCH'
    END AS match_type,

    CASE
        WHEN
            COALESCE(pl."Part. Brasil", 0)
          + COALESCE(pl."Part. Usa", 0)
          + COALESCE(pl."Part. Europa", 0) > 0
        THEN
            COALESCE(pf.precio_br, 0)  * COALESCE(pl."Part. Brasil", 0)
          + COALESCE(pf.precio_usa, 0) * COALESCE(pl."Part. Usa", 0)
          + COALESCE(pf.precio_eur, 0) * COALESCE(pl."Part. Europa", 0)

        WHEN pf.precio_br IS NULL
         AND pf.precio_usa IS NULL
        THEN NULL

        WHEN pf.precio_br IS NULL
        THEN pf.precio_usa

        WHEN pf.precio_usa IS NULL
        THEN pf.precio_br

        ELSE
            (pf.precio_br * 0.5)
          + (pf.precio_usa * 0.5)
    END AS "Precio Prorrateo"

FROM precios_lista pl
LEFT JOIN precio_familia pf
    ON pl.ref_lista = pf.Referencia_Principal;
""")

print("✅ resultado_precios_lista creada")

# =========================================================
# 6. EXPORTAR A EXCEL
# =========================================================
df_out = duck.execute("SELECT * FROM resultado_precios_lista").df()
df_out.to_excel(EXCEL_OUT, index=False)

print(f"📂 Excel generado: {EXCEL_OUT}")

# =========================================================
# 7. CIERRE
# =========================================================
duck.close()
print("✅ Proceso finalizado correctamente")

✅ DuckDB conectado y bases adjuntadas
✅ precio_familia creada
✅ Precios de Lista cargados
✅ resultado_precios_lista creada
📂 Excel generado: Precio_De_Lista_con_Precios.xlsx
✅ Proceso finalizado correctamente
